
# AHS-KT × ASSIST2009 最小可运行 Notebook

这本 Notebook 的目标是：

1. 使用 `/root/autodl-tmp/ahs-kt` 项目代码；
2. 使用仓库已经准备好的 `assist2009` 数据；
3. 把 `AHS-KT + ASSIST2009` 的训练与评估完整跑通；
4. 输出 `acc / auc / f1`。

## 为什么这次不用自己重建数据

和 `junyi` 不一样，当前仓库里已经自带了 `assist2009` 的预处理结果：
- `data/assist2009_train_ahskt.npz`
- `data/assist2009_valid_ahskt.npz`
- `data/assist2009_test_ahskt.npz`
- 以及更完整的 `assist2009_v5_*` 版本

所以从第一性原理看，最短路径不是重新 build，而是：
- 直接复用现成 bundle；
- 复用仓库现成的 `assist2009_v5` 配置；
- 只在 Notebook 里把输出目录改成 notebook 专用路径，避免和已有结果混在一起。

## 本 Notebook 采用的配置

默认基于：
- `configs/ahskt_assist2009_v5.json`

并在 Notebook 里重写成：
- 专用 `task_name`
- 专用 `output_root`

这样可以做到：
- 路径最短；
- 训练可复现；
- 不污染原来的实验目录。


In [1]:

from pathlib import Path
import os
import sys
import json
import time
import random

PROJECT_ROOT = Path('/root/autodl-tmp/ahs-kt')
SRC_ROOT = PROJECT_ROOT / 'src'
BASE_CONFIG_PATH = PROJECT_ROOT / 'configs/ahskt_assist2009_v5.json'
METADATA_PATH = PROJECT_ROOT / 'data/assist2009_metadata.json'

assert PROJECT_ROOT.exists(), f'找不到项目目录: {PROJECT_ROOT}'
assert BASE_CONFIG_PATH.exists(), f'找不到基础配置: {BASE_CONFIG_PATH}'
assert METADATA_PATH.exists(), f'找不到 assist2009 元数据: {METADATA_PATH}'

os.chdir(PROJECT_ROOT)
os.environ.setdefault('OMP_NUM_THREADS', '1')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('BASE_CONFIG_PATH =', BASE_CONFIG_PATH)
print('METADATA_PATH =', METADATA_PATH)
print('OMP_NUM_THREADS =', os.environ['OMP_NUM_THREADS'])


PROJECT_ROOT = /root/autodl-tmp/ahs-kt
BASE_CONFIG_PATH = /root/autodl-tmp/ahs-kt/configs/ahskt_assist2009_v5.json
METADATA_PATH = /root/autodl-tmp/ahs-kt/data/assist2009_metadata.json
OMP_NUM_THREADS = 1


In [2]:

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

from ahskt.config import load_config
from ahskt.data.dataset import load_bundle_from_config
from ahskt.models.ahs_kt import AHSKTModel
from ahskt.training.engine import fit_and_evaluate

print('TensorFlow version =', tf.__version__)
print('GPU devices =', tf.config.list_physical_devices('GPU'))
for gpu_device in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu_device, True)
    except RuntimeError:
        pass


TensorFlow version = 2.8.0
GPU devices = [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    assist2009_metadata = json.load(f)

print(json.dumps(assist2009_metadata, ensure_ascii=False, indent=2))


{
  "dataset_name": "assist2009",
  "sequence_length": 100,
  "remainder_min_len": 10,
  "num_questions": 9798,
  "num_concepts": 107,
  "num_question_difficulty": 101,
  "num_concept_difficulty": 101,
  "num_behavior_clusters": 5,
  "split_summary": {
    "train_users": 3375,
    "valid_users": 421,
    "test_users": 421,
    "train_sequences": 3811,
    "valid_sequences": 492,
    "test_sequences": 500
  },
  "default_difficulty": 66
}



## Notebook 专用配置

这里直接读取 `ahskt_assist2009_v5.json`，然后只改两件事：
- `task_name`
- `outputs.root_dir`

模型超参数、数据路径和训练超参数都保持和 `v5` 一致。


In [4]:

SEED = 2026
TASK_NAME = 'ahskt_assist2009_v5_notebook'
NOTEBOOK_CONFIG_PATH = PROJECT_ROOT / 'configs/ahskt_assist2009_v5_notebook.json'
NOTEBOOK_OUTPUT_ROOT = PROJECT_ROOT / 'outputs/assist2009_v5_notebook_run'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

with open(BASE_CONFIG_PATH, 'r', encoding='utf-8') as f:
    config_payload = json.load(f)

config_payload['seed'] = SEED
config_payload['task_name'] = TASK_NAME
config_payload['outputs']['root_dir'] = str(NOTEBOOK_OUTPUT_ROOT.relative_to(PROJECT_ROOT))

NOTEBOOK_CONFIG_PATH.write_text(json.dumps(config_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved notebook config to', NOTEBOOK_CONFIG_PATH)
print(json.dumps(config_payload, ensure_ascii=False, indent=2))


saved notebook config to /root/autodl-tmp/ahs-kt/configs/ahskt_assist2009_v5_notebook.json
{
  "project_name": "ahs-kt",
  "task_name": "ahskt_assist2009_v5_notebook",
  "seed": 2026,
  "dataset": {
    "mode": "real_npz",
    "train_path": "data/assist2009_v5_train_ahskt.npz",
    "valid_path": "data/assist2009_v5_valid_ahskt.npz",
    "test_path": "data/assist2009_v5_test_ahskt.npz"
  },
  "model": {
    "num_questions": 9798,
    "num_concepts": 107,
    "num_question_difficulty": 98,
    "num_concept_difficulty": 91,
    "num_behavior_clusters": 5,
    "sequence_length": 100,
    "embedding_dim": 64,
    "difficulty_dim": 32,
    "behavior_dim": 32,
    "hidden_dim": 96,
    "dropout": 0.2,
    "use_behavior_cluster": true,
    "use_difficulty_features": true,
    "use_behavior_features": true,
    "use_target_interaction": true,
    "question_global_easiness": 0.6536111192441055,
    "concept_global_easiness": 0.6536111192441055,
    "fusion_mode": "late_residual",
    "behavior_c

In [5]:

config = load_config(NOTEBOOK_CONFIG_PATH, project_root=PROJECT_ROOT)
train_bundle, valid_bundle, test_bundle = load_bundle_from_config(config)

print('train / valid / test 序列数 =', train_bundle.num_samples, valid_bundle.num_samples, test_bundle.num_samples)
print('sequence_length =', train_bundle.sequence_length)
print('train bundle shapes:')
for key, value in train_bundle.as_dict().items():
    print(f'  {key}: {value.shape}')


train / valid / test 序列数 = 3811 492 500
sequence_length = 100
train bundle shapes:
  question_ids: (3811, 100)
  concept_ids: (3811, 100)
  responses: (3811, 100)
  question_difficulty: (3811, 100)
  concept_difficulty: (3811, 100)
  attempts: (3811, 100)
  hints: (3811, 100)
  speed: (3811, 100)
  behavior_cluster: (3811, 100)
  mask: (3811, 100)
  question_easiness: (3811, 100)
  concept_easiness: (3811, 100)
  question_confidence: (3811, 100)
  concept_confidence: (3811, 100)



## 开始训练 AHS-KT

下面这段代码和项目脚本 `scripts/train_ahskt.py` 的核心逻辑保持一致：
- `load_config`
- `load_bundle_from_config`
- `AHSKTModel`
- `fit_and_evaluate`

唯一的区别是：这里直接在 Notebook 里执行，方便你查看中间结果。


In [6]:

model = AHSKTModel(config.model)

train_start = time.time()
metrics_summary = fit_and_evaluate(
    model=model,
    train_bundle=train_bundle,
    valid_bundle=valid_bundle,
    test_bundle=test_bundle,
    config=config,
)
train_elapsed = time.time() - train_start

metrics_path = config.output_root / f'{config.task_name}_metrics.json'
metrics_path.write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('训练耗时(秒) =', round(train_elapsed, 2))
print('metrics_path =', metrics_path)
print(json.dumps(metrics_summary, ensure_ascii=False, indent=2))


2026-04-07 10:51:35.396430: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-07 10:51:36.008688: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22182 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:57:00.0, compute capability: 8.6
2026-04-07 10:51:36.848134: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-04-07 10:51:37.771420: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8200


训练耗时(秒) = 40.9
metrics_path = /root/autodl-tmp/ahs-kt/outputs/assist2009_v5_notebook_run/ahskt_assist2009_v5_notebook_metrics.json
{
  "task_name": "ahskt_assist2009_v5_notebook",
  "seed": 2026,
  "best_epoch": 2,
  "best_valid_auc": 0.7839445036418405,
  "test_metrics": {
    "loss": 0.5329184532165527,
    "auc": 0.7817715751303898,
    "acc": 0.7345913569910478,
    "rmse": 0.4222089350223541
  },
  "history": [
    {
      "epoch": 1,
      "train": {
        "loss": 0.5432170033454895,
        "auc": 0.7541209009657963,
        "acc": 0.7295662347792174,
        "rmse": 0.42698314785957336
      },
      "valid": {
        "loss": 0.5286592245101929,
        "auc": 0.778148169423234,
        "acc": 0.7360407771489094,
        "rmse": 0.42035356163978577
      }
    },
    {
      "epoch": 2,
      "train": {
        "loss": 0.4852598011493683,
        "auc": 0.8160187247521571,
        "acc": 0.7630237899207003,
        "rmse": 0.4007185399532318
      },
      "valid": {
       


## 补充 F1 指标

项目原生 `metrics.py` 默认输出：
- `loss`
- `auc`
- `acc`
- `rmse`

这里我们额外基于测试集输出 `f1`，方便直接汇报。


In [7]:

def collect_targets_and_predictions(model, bundle, batch_size):
    dataset = bundle.to_tf_dataset(batch_size=batch_size, shuffle=False)
    all_targets = []
    all_predictions = []
    for batch in dataset:
        logits = model(batch, training=False)
        next_logits = logits[:, :-1]
        next_targets = tf.cast(batch['responses'][:, 1:], tf.float32)
        next_mask = tf.cast(batch['mask'][:, 1:], tf.float32)
        valid_logits = tf.boolean_mask(next_logits, next_mask > 0)
        valid_targets = tf.boolean_mask(next_targets, next_mask > 0)
        all_targets.append(valid_targets.numpy())
        all_predictions.append(tf.sigmoid(valid_logits).numpy())
    return np.concatenate(all_targets, axis=0), np.concatenate(all_predictions, axis=0)


test_targets, test_predictions = collect_targets_and_predictions(
    model=model,
    bundle=test_bundle,
    batch_size=config.training.batch_size,
)

test_binary_predictions = (test_predictions > 0.5).astype(int)
summary_with_f1 = {
    'acc': float(accuracy_score(test_targets, test_binary_predictions)),
    'auc': float(roc_auc_score(test_targets, test_predictions)),
    'f1': float(f1_score(test_targets, test_binary_predictions)),
    'loss': float(metrics_summary['test_metrics']['loss']),
    'rmse': float(metrics_summary['test_metrics']['rmse']),
    'num_test_points': int(len(test_targets)),
}

summary_with_f1_path = config.output_root / f'{config.task_name}_metrics_with_f1.json'
summary_with_f1_path.write_text(json.dumps(summary_with_f1, ensure_ascii=False, indent=2), encoding='utf-8')

print('summary_with_f1_path =', summary_with_f1_path)
print(json.dumps(summary_with_f1, ensure_ascii=False, indent=2))


summary_with_f1_path = /root/autodl-tmp/ahs-kt/outputs/assist2009_v5_notebook_run/ahskt_assist2009_v5_notebook_metrics_with_f1.json
{
  "acc": 0.7345913569910478,
  "auc": 0.7817715751303898,
  "f1": 0.805512294114871,
  "loss": 0.5329184532165527,
  "rmse": 0.4222089350223541,
  "num_test_points": 31054
}



## 结论

如果你只是想“把 AHS-KT + ASSIST2009 跑通”，这本 Notebook 已经完成了：
- 数据：使用仓库预处理好的 `assist2009_v5` bundle
- 模型：`AHSKTModel`
- 训练：使用项目原生训练循环
- 结果：输出 `acc / auc / f1`

如果你下一步要做更正式的实验，可以继续：
- 固定多随机种子重复训练
- 对比 `assist2009_v1` / `v3` / `v5`
- 跑 ablation 配置
- 汇总多次实验均值与方差
